# task 4

update nov 3

before: we used the organizers' baseline file, which is fit to the train months' link counts.
now: we use the file fit to the counts of the month being scored (validation and private each use their own counts, released with the data). the link-count score goes from about 0.965 to 0.9999 on validation.

same format as before: odme.csv, 70,708 rows. internet on.


In [ ]:
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
REL = Path("/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public")
REPO = Path("/tmp/trafficflowbench-public")
OUT = Path("/kaggle/working")


def run(*args):
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    r.check_returncode()
    return r.stdout


if not REPO.exists():
    run("git", "clone", "https://github.com/jacky850/trafficflowbench-public.git", REPO)
print("organizers' repo commit:", run("git", "-C", REPO, "rev-parse", "HEAD").strip())


def stack(parts, dest):
    with open(dest, "w") as out:
        for i, p in enumerate(parts):
            with open(p) as f:
                header = f.readline()
                if i == 0:
                    out.write(header)
                out.writelines(f)


def count_rows(path):
    with open(path) as f:
        return sum(1 for _ in f) - 1

In [ ]:
for split in ["validation", "private"]:
    for p in sorted(x.name for x in (REL / "task4").iterdir() if x.is_dir()):
        assert (REL / "task4" / p / split / "synthetic_link_counts.csv").exists(), (split, p)

EXPECTED_ROWS = 35_354

parts = []
for split in ["validation", "private"]:
    root = Path(f"/tmp/task4_{split}")
    run("python", REPO / "src/task4/build_task4_odme_artifacts.py",
        "--release-root", REL, "--split", split, "--output-root", root)
    f = root / "public_reference.csv"
    assert count_rows(f) == EXPECTED_ROWS, (split, count_rows(f))
    parts.append(f)

stack(parts, OUT / "odme.csv")
print("odme.csv rows:", count_rows(OUT / "odme.csv"))

S_link, own-month counts (validation and private)

In [ ]:
for split in ["validation", "private"]:
    run("python", REPO / "src/task4/score_task4.py",
        "--submission", Path(f"/tmp/task4_{split}/public_reference.csv"),
        "--release-root", REL, "--split", split)